# VERITAS · CPU evidence verification
Index two public-source excerpts, check a real passage, and reproduce a controlled duplicate experiment. The notebook imports the release implementation. A fresh Colab runtime downloads the repository and its Python dependencies.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "src/veritas").exists():
    if (root.parent / "src/veritas").exists():
        root = root.parent
    else:
        root = Path("/content/veritas") if Path("/content").exists() else root / "veritas"
        if not root.exists():
            subprocess.run(
                [
                    "git",
                    "clone",
                    "--depth",
                    "1",
                    "--branch",
                    "v0.1.0",
                    "https://github.com/Rahulchaube1/veritas.git",
                    str(root),
                ],
                check=True,
            )
os.chdir(root)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
sys.path.insert(0, str(root / "src"))
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

In [ ]:
import json

from veritas import Veritas

engine = Veritas()
engine.add([json.loads(line) for line in Path("datasets/public_examples.jsonl").read_text().splitlines()])
result = engine.search("Neptune is the eighth and most distant planet in our solar system.")
print(result.answer)
for i in result.claims[0].citations:
    passage = result.evidence[i]
    print(passage.passage, passage.url)
assert result.claims[0].verdict == "supported"

## Duplicate injection
The same opposing report appears on eight nominally different pages. Compare the two aggregation rules on the same retrieved documents.

In [ ]:
from veritas import Config

copies = Veritas(Config(page_size=12))
claim = "The ferry carries 40 passengers."
copies.add(
    [{"id": "primary", "text": claim}]
    + [{"id": f"copy-{n}", "text": "The ferry carries 41 passengers."} for n in range(8)]
)
for grouped in (True, False):
    r = copies.search(claim, dependency=grouped)
    print("grouped", grouped, "verdict", r.claims[0].verdict, "balance", round(r.evidence_balance, 3))
assert copies.search(claim).claims[0].verdict == "conflict"

## Reproduce the synthetic benchmark
This cell regenerates all 1,440 cases, fits the synthetic calibrator and scores the 360 designated test cases.

In [ ]:
subprocess.run([sys.executable, "experiments/controlled.py"], check=True)
metrics = json.loads(Path("results/controlled.json").read_text())
print(metrics["ablations"]["full"])
print(metrics["families"])

## Open the search interface in Colab
Set LAUNCH_UI to True in a Colab runtime. The application runs in that runtime and uses the two public excerpts already loaded above.

In [ ]:
LAUNCH_UI = False
if LAUNCH_UI:
    import threading

    import uvicorn
    from google.colab import output

    from veritas.api import create_app

    app = create_app(engine)
    threading.Thread(
        target=uvicorn.run, args=(app,), kwargs={"host": "127.0.0.1", "port": 8000}, daemon=True
    ).start()
    output.serve_kernel_port_as_iframe(8000, height=760)

The reproducibility guide covers the official SciFact archive, optional pretrained models and learned scientific rationale filter. The default CPU demonstration does not evaluate open-domain factual accuracy.